# 벡터 DB 생성 및 적재를 위한 노트북 파일입니다.

### step1 - 필요한 데이터 경로 가져오기

In [1]:
from pathlib import Path

import pandas as pd
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# Load the CSV from the notebook parent directory.
csv_path = Path("../museum_final_738.csv")
df = pd.read_csv(csv_path, encoding="utf-8-sig")

#테스트
display(df.head(2))


,museum_id,name,ownership_type,sido,sigungu,address,latitude,longitude,operator_name,manager_name,...,official_verified_field_count,text,text_length,text_sha256,source_doc_count,source_types,source_documents_json,text_spans_json,removed_duplicate_sentence_count,merge_review_flags
0,M0001,박진전쟁기념관,공립,경상남도,창녕군,경상남도 창녕군 남지읍 월상길 27,35.447968,128.386193,경상남도 창녕군청,경상남도 창녕군청,...,0.0,박진전쟁기념관은 6·25전쟁 최후의 보루였던 박진전투 현장을 재현하여 민족 분단의 ...,407,1fcf9c09a671bb09df9f0a8d84edf4d13a2fd68b08c0c4...,1,TOUR_API,"[{""doc_id"":""DOC-B0001-TOUR_API-2785601"",""sourc...","[{""start"":0,""end"":94,""doc_ids"":[""DOC-B0001-TOU...",0,NaN
1,M0003,창녕박물관,공립,경상남도,창녕군,경상남도 창녕군 창녕읍 창밀로 34,35.545406,128.505441,경상남도 창녕군청,경상남도 창녕군청,...,0.0,창녕박물관은 1996년 창녕지역 고분군에서 출토된 유물들을 전시할 목적으로 창녕유물...,608,35480738914dd66b95a1bef1f432ae7ce951e01ee9503d...,1,TOUR_API,"[{""doc_id"":""DOC-B0003-TOUR_API-130107"",""source...","[{""start"":0,""end"":60,""doc_ids"":[""DOC-B0003-TOU...",0,NaN


### step2 - 임베딩 모델 가져오기(text-embedding-3-small)

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

env_path = next((parent / ".env" for parent in [Path.cwd(), *Path.cwd().parents] if (parent / ".env").is_file()), None)
if env_path:
    load_dotenv(env_path)
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Set OPENAI_API_KEY in .env or the environment.")

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")


### step3 - 벡터DB에 필요한 열만 남기기

In [3]:
# ==== ChromaDB 적재용 데이터 준비 ====

# 1. 사용할 열 선택
chroma_columns = [
    # 식별자 및 임베딩 본문
    "museum_id",
    "text",

    # 시설 기본 정보
    "name",
    "ownership_type",
    "sido",
    "sigungu",
    "address",
    "latitude",
    "longitude",
    "homepage_url",

    # 관람료 및 조건 검색
    "adult_fee",
    "youth_fee",
    "child_fee",
    "adult_fee_status",
    "fee_notes",

    # 운영 정보
    "weekday_open",
    "weekday_close",
    "holiday_open",
    "holiday_close",
    "closure_rule_raw",

    # 편의시설 및 안내
    "amenities_raw",
    "transport_notes",
    "image_url",
    "image_copyright",

    # 출처 및 품질 관리
    "source_types",
    "requires_manual_review",
]

# 2. 필요한 열만 추출 (원본 df 유지)
df_chroma = df[chroma_columns].copy()

# 3. 준비된 데이터 저장
df_chroma.to_csv(
    "museum_chroma_ready.csv",
    index=False,
    encoding="utf-8-sig"
)

# 4. 상위 2개 행 확인
display(df_chroma.head(2))

,museum_id,text,name,ownership_type,sido,sigungu,address,latitude,longitude,homepage_url,...,weekday_close,holiday_open,holiday_close,closure_rule_raw,amenities_raw,transport_notes,image_url,image_copyright,source_types,requires_manual_review
0,M0001,박진전쟁기념관은 6·25전쟁 최후의 보루였던 박진전투 현장을 재현하여 민족 분단의 ...,박진전쟁기념관,공립,경상남도,창녕군,경상남도 창녕군 남지읍 월상길 27,35.447968,128.386193,https://www.cng.go.kr/01656/03204/03205.web,...,18:00,09:00,18:00,매주 월요일+1월 1일+설날+추석날,장애인화장실+유모차/휠체어 이용가능,NaN,https://tong.visitkorea.or.kr/cms/resource/99/...,Type1,TOUR_API,True
1,M0003,창녕박물관은 1996년 창녕지역 고분군에서 출토된 유물들을 전시할 목적으로 창녕유물...,창녕박물관,공립,경상남도,창녕군,경상남도 창녕군 창녕읍 창밀로 34,35.545406,128.505441,https://www.cng.go.kr/01656/01671.web,...,18:00,09:00,18:00,매주 월요일+1월 1일+설날+추석날,장애인화장실+유모차/휠체어 대여+유모차/휠체어 이용가능,NaN,https://tong.visitkorea.or.kr/cms/resource/15/...,Type1,TOUR_API,True


### step4 - text 열 임베딩 처리하기 + 적재

- museum_id만 메타데이터가 아니라 ChromaDB의 고유 ID로 저장 (ids로 저장됨)

In [5]:

# ==== ChromaDB 최종 구축: name + text ====

import numpy as np
import pandas as pd
import chromadb

COLLECTION_NAME = "museum_documents"

# 프로젝트 루트에 있는 data/vector_db 폴더 사용
PROJECT_ROOT = Path.cwd()

# 노트북 위치에서 실행 중이라면 프로젝트 루트로 이동
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = Path.cwd().resolve().parents[2]

DB_PATH = PROJECT_ROOT / "data" / "vector_db" / "museum_chroma"
DB_PATH.mkdir(parents=True, exist_ok=True)

print(f"ChromaDB 저장 경로: {DB_PATH}")

# 1. 데이터 준비
df_ready = df_chroma.copy()

assert df_ready["museum_id"].notna().all()
assert df_ready["museum_id"].is_unique
assert df_ready["text"].notna().all()
assert df_ready["name"].notna().all()

ids = df_ready["museum_id"].astype(str).tolist()
documents = df_ready["text"].astype(str).tolist()

# 2. 임베딩 입력: name + text
embedding_inputs = [
    f"박물관명: {name}\n박물관 설명: {text}"
    for name, text in zip(
        df_ready["name"],
        documents
    )
]

# 3. 메타데이터 준비
def make_metadata(row):
    metadata = {}

    for col, value in row.items():
        if col in ("museum_id", "text"):
            continue

        if pd.isna(value):
            continue

        if isinstance(value, str):
            value = value.strip()
            if not value:
                continue

        if col in ("latitude", "longitude",
                   "adult_fee", "youth_fee", "child_fee"):
            value = pd.to_numeric(value, errors="coerce")
            if pd.isna(value) or not np.isfinite(value):
                continue
            value = float(value)

        elif col == "requires_manual_review":
            if isinstance(value, str):
                if value.lower() not in ("true", "false"):
                    continue
                value = value.lower() == "true"
            else:
                value = bool(value)

        elif isinstance(value, (np.bool_, bool)):
            value = bool(value)

        elif isinstance(value, (np.integer, int)):
            value = int(value)

        elif isinstance(value, (np.floating, float)):
            value = float(value)

        else:
            value = str(value)

        metadata[col] = value

    return metadata

metadatas = [
    make_metadata(row)
    for _, row in df_ready.iterrows()
]

# 4. 기존 저장 경로에 새 Collection 생성
client = chromadb.PersistentClient(path=str(DB_PATH))

collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={
        "hnsw:space": "cosine",
        "embedding_strategy": "name_plus_text",
        "embedding_model": "text-embedding-3-small"
    }
)

if collection.count() != 0:
    raise RuntimeError(
        "기존 Collection에 데이터가 남아 있습니다. "
        "삭제 후 다시 실행하세요."
    )

# 5. 임베딩 생성
vectors = embeddings.embed_documents(embedding_inputs)

# 6. ChromaDB 적재
collection.add(
    ids=ids,
    documents=documents,        # 원문 text 유지
    embeddings=vectors,         # name + text 임베딩
    metadatas=metadatas
)

# 7. 저장 결과 검증
check_client = chromadb.PersistentClient(path=str(DB_PATH))
check_collection = check_client.get_collection(COLLECTION_NAME)

assert check_collection.count() == len(df_ready)

result = check_collection.get(
    ids=ids[:2],
    include=["documents", "metadatas", "embeddings"]
)

assert len(result["ids"]) == 2

print("ChromaDB 최종 구축 성공")
print(f"저장 위치: {DB_PATH}")
print(f"저장 문서: {check_collection.count()}건")
print(f"임베딩 차원: {len(result['embeddings'][0])}")
print("임베딩 방식: name + text")

display(pd.DataFrame([
    {
        "museum_id": mid,
        "name": meta.get("name"),
        "text": doc[:100],
        "embedding_dim": len(vec)
    }
    for mid, meta, doc, vec in zip(
        result["ids"],
        result["metadatas"],
        result["documents"],
        result["embeddings"]
    )
]).head(2))


ChromaDB 저장 경로: C:\Users\Playdata\Desktop\mle-01-p3-team1\data\vector_db\museum_chroma
ChromaDB 최종 구축 성공
저장 위치: C:\Users\Playdata\Desktop\mle-01-p3-team1\data\vector_db\museum_chroma
저장 문서: 738건
임베딩 차원: 1536
임베딩 방식: name + text


,museum_id,name,text,embedding_dim
0,M0001,박진전쟁기념관,박진전쟁기념관은 6·25전쟁 최후의 보루였던 박진전투 현장을 재현하여 민족 분단의 ...,1536
1,M0003,창녕박물관,창녕박물관은 1996년 창녕지역 고분군에서 출토된 유물들을 전시할 목적으로 창녕유물...,1536


### step5 - 저장된 DB 확인

In [6]:
# ==== 저장된 ChromaDB 재조회 및 적재 검증 ====

from IPython.display import display

# 1. 디스크에 저장된 DB에 다시 연결
check_client = chromadb.PersistentClient(
    path=str(DB_PATH)
)

check_collection = check_client.get_collection(
    name=COLLECTION_NAME
)

# 2. 적재 건수 및 ID 검증
stored_count = check_collection.count()

assert stored_count == len(df_ready), (
    f"건수 불일치: {stored_count} / {len(df_ready)}"
)

stored_ids = set(check_collection.get(include=[])["ids"])
assert stored_ids == set(ids), "저장된 museum_id가 원본과 다릅니다."

# 3. 첫 2개 문서와 임베딩 조회
result = check_collection.get(
    ids=ids[:2],
    include=["documents", "metadatas", "embeddings"]
)

assert len(result["ids"]) == 2
assert result["embeddings"] is not None

# 4. 확인용 테이블
check_df = pd.DataFrame([
    {
        "museum_id": doc_id,
        "name": meta.get("name"),
        "text": doc[:100],
        "embedding_dim": len(vector)
    }
    for doc_id, doc, meta, vector in zip(
        result["ids"],
        result["documents"],
        result["metadatas"],
        result["embeddings"]
    )
])

print(f"저장 위치: {DB_PATH}")
print(f"저장된 문서: {stored_count}건")
print(f"임베딩 차원: {len(result['embeddings'][0])}")
print("ChromaDB 재조회 및 적재 검증 성공")

display(check_df.head(2))

저장 위치: C:\Users\Playdata\Desktop\mle-01-p3-team1\data\vector_db\museum_chroma
저장된 문서: 738건
임베딩 차원: 1536
ChromaDB 재조회 및 적재 검증 성공


,museum_id,name,text,embedding_dim
0,M0001,박진전쟁기념관,박진전쟁기념관은 6·25전쟁 최후의 보루였던 박진전투 현장을 재현하여 민족 분단의 ...,1536
1,M0003,창녕박물관,창녕박물관은 1996년 창녕지역 고분군에서 출토된 유물들을 전시할 목적으로 창녕유물...,1536


### step 6 - 검색으로 테스트 해보기

In [7]:
# ==== ChromaDB 의미 검색 테스트 (Top 3) ====

# 1. 사용자 질문 입력
query = input("검색할 질문을 입력하세요: ").strip()

if not query:
    raise ValueError("검색 질문을 입력해야 합니다.")

# 2. 질문 임베딩
query_vector = embeddings.embed_query(query)

# 3. ChromaDB 유사도 검색
results = check_collection.query(
    query_embeddings=[query_vector],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

# 4. 원래 질문 출력
print("\n" + "=" * 70)
print(f"🔎 원래 질문: {query}")
print("=" * 70)

# 5. 상위 3개 검색 결과 출력
for i, (museum_id, document, metadata, distance) in enumerate(
    zip(
        results["ids"][0],
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ),
    start=1
):
    metadata = metadata or {}

    print(f"\n{'=' * 70}")
    print(f"📌 검색 결과 {i}위")
    print(f"{'=' * 70}")

    print(f"박물관명 : {metadata.get('name', '정보 없음')}")
    print(f"박물관 ID: {museum_id}")
    print(f"지역     : {metadata.get('sido', '')} {metadata.get('sigungu', '')}")
    print(f"벡터 거리: {distance:.4f}")

    print("\n[검색된 원본 텍스트]")
    print(document)

print(f"\n✅ 총 {len(results['ids'][0])}개 검색 완료")

ValueError: 검색 질문을 입력해야 합니다.

In [ ]:
# ==== 벡터 검색 Top 10 및 원본 데이터 일치 검증 ====

import pandas as pd
from IPython.display import display

# 기존 질문 벡터로 상위 10개 조회
result = check_collection.query(
    query_embeddings=[query_vector],
    n_results=10,
    include=["documents", "metadatas", "distances"]
)

# 원본 데이터 매핑
original = df_chroma.set_index("museum_id")

rows = []

for i, (mid, doc, meta, dist) in enumerate(zip(
    result["ids"][0],
    result["documents"][0],
    result["metadatas"][0],
    result["distances"][0]
), start=1):

    # 원본 CSV와 저장된 본문 비교
    is_matched = (
        mid in original.index
        and original.loc[mid, "text"] == doc
        and original.loc[mid, "name"] == meta.get("name")
    )

    rows.append({
        "순위": i,
        "museum_id": mid,
        "박물관명": meta.get("name"),
        "거리": round(float(dist), 4),
        "원본 일치": is_matched,
        "본문 미리보기": doc[:100].replace("\n", " ")
    })

print(f"검색 질문: {query}")
display(pd.DataFrame(rows))

검색 질문: 일제강점기 독립운동과 관련된 박물관


,순위,museum_id,박물관명,거리,원본 일치,본문 미리보기
0,1,M0875,몽양기념관,0.9219,True,2011년 11월 27일에 개관. 독립 ·평화통일운동가 여운형 관련 역사박물관
1,2,M0191,경상북도독립운동기념관,1.0089,True,경상북도독립운동기념관은 2007년 개관한 안동독립운동기념관을 2014년에 경상북도 ...
2,3,M0519,국립낙동강생물자원관,1.0385,True,국립낙동강생물자원관은 담수(淡水) 분야에서 국가 생물주권을 확보하고 생물 다양성의 ...
3,4,M0525,국립관세박물관,1.0758,True,"국립관세박물관은 강남구에 위치한 국립박물관으로, 개항기부터 현대까지 한국 경제사와 ..."
4,5,M0840,경기도박물관,1.0765,True,경기도어린이박물관과 이웃하고 있는 경기도박물관은 경기도의 역사와 전통문화를 계승하고...
5,6,M0502,국립일제강제동원역사관,1.0772,True,국립일제강제동원역사관은 일제 강점기 강제동원의 실상을 규명함으로써 성숙한 역사의식을...
6,7,M0453,경북대학교 박물관,1.0780,True,"경북대학교 박물관은 1959년 개관 이래 유물의 수집과 보존, 전시, 연구 등을 통..."
7,8,M0930,강릉자수박물관,1.0821,True,강릉 오죽헌의 예술창작인촌에 위치한 강릉자수박물관은 자수에 새겨진 옛 여인들의 아름...
8,9,M0363,한림대학교박물관,1.0894,True,"한림대학교 박물관은 선사와 역사시대의 유물과 민속품 등을 수집 정리하고, 보전과 전..."
9,10,M0151,밀양시립박물관,1.0905,True,"밀양시립박물관은 낙동강유역의 문화를 집대성하여 효율적인 보존·전시함은 물론, 향토사..."
